<a href="https://colab.research.google.com/github/jacielefreitas63-tech/projeto-delivery-logistic/blob/main/1_limpeza_e_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Projeto Delivery: Engenharia de Dados & Análise Exploratória (EDA)


# 1 Configuração do Ambiente e Conexão com o Data Lake (Google Drive)

Neste notebook, realizaremos a ingestão, limpeza fina, tratamento de outliers e união das bases transacionais da *Olist, malha rodoviária do **OpenStreetMap* e dados meteorológicos do *INMET*.

In [1]:
!pip install osmnx networkx pandas numpy pyspark

In [2]:
# Importação das bibliotecas fundamentais para manipulação e análise estatística
import pandas as pd
import numpy as np
import networkx as nx
import os as ox

# Imports específicos do PySpark para processamento distribuído
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
print("🚀 Bibliotecas carregadas com sucesso!")

🚀 Bibliotecas carregadas com sucesso!


In [3]:
# Inicialização do motor do Spark
spark = SparkSession.builder \
    .appName("LogisticaDelivery") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("🚀 Spark inicializado com sucesso e pronto para processamento distribuído!")

🚀 Spark inicializado com sucesso e pronto para processamento distribuído!


In [4]:
# Conexão segura com o armazenamento de dados do Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 📦2.Ingestão da Base de Pedidos (Olist Orders)
> O objetivo desta etapa é ler o dataset transacional de ordens de serviço, inspecionar os tipos de dados primitivos e mapear o volume inicial de registros antes de aplicar as regras de limpeza.

In [5]:
# Caminho
base_path = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

In [ ]:
import pandas as pd
import os

# Mapeamento dos arquivos
arquivos = {
    'customers': 'olist_customers_dataset.csv',
    'items': 'olist_order_items_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv'
}

df_raw = {}

# Carregamento e Otimização em Tempo de Execução
for nome, arquivo in arquivos.items():
    caminho_completo = os.path.join(base_path, arquivo)

    if os.path.exists(caminho_completo):
        # Carrega o CSV
        df_raw[nome] = pd.read_csv(caminho_completo)


        print(f"✅ '{nome}' processado. Shape: {df_raw[nome].shape}")
    else:
        print(f"❌ Arquivo não encontrado: {arquivo}")

In [ ]:
 # Otimização: Converte objetos de baixa cardinalidade para 'category'
for col in df_raw[nome].select_dtypes(include=['object']).columns:
  if df_raw[nome][col].nunique() < 1000:
    df_raw[nome][col] = df_raw[nome][col].astype('category')

In [ ]:
  # Otimização: Downcast de numéricos para reduzir footprint de memória
cols_num = df_raw[nome].select_dtypes(include=['int64', 'float64']).columns
for col in cols_num:
    if df_raw[nome][col].dtype == 'int64':
        df_raw[nome][col] = pd.to_numeric(df_raw[nome][col], downcast='integer')
    else:
        df_raw[nome][col] = pd.to_numeric(df_raw[nome][col], downcast='float')

print(f"✅ '{nome}' processado. Shape: {df_raw[nome].shape}")

Tratamento Individual (Limpeza pré-merge)
Tratamos o CEP aqui para garantir que o merge funcione perfeitamente.

In [ ]:
# Tratamento de CEP (String com 5 dígitos)
df_raw['customers']['customer_zip_code_prefix'] = df_raw['customers']['customer_zip_code_prefix'].astype(str).str.zfill(5)
df_raw['geolocation']['geolocation_zip_code_prefix'] = df_raw['geolocation']['geolocation_zip_code_prefix'].astype(str).str.zfill(5)

# Criando versão única da geolocalização para evitar duplicatas no merge
df_geo_unica = df_raw['geolocation'].drop_duplicates(subset=['geolocation_zip_code_prefix'])

In [ ]:
# --- Mapeamento das bases de dados ---
# Extração das tabelas do dicionário 'df_raw' para variáveis individuais.
# Isso garante a compatibilidade com o pipeline de consolidação abaixo.

# It is assumed that 'df_raw' (a dictionary of Pandas DataFrames)
# and 'df_geo_unica' (a Pandas DataFrame) are defined by prior cells.
# The error 'NameError: name 'df_raw' is not defined' indicates that these
# prior cells were not executed or failed.

# Convert Pandas DataFrames to PySpark DataFrames for downstream operations
df_pedidos = spark.createDataFrame(df_raw['orders'])
df_clientes = spark.createDataFrame(df_raw['customers'])
df_reviews = spark.createDataFrame(df_raw['reviews'])

# For df_geolocation, convert to PySpark and rename columns for consistency
spark_df_geolocation_base = spark.createDataFrame(df_geo_unica)
df_geolocation = spark_df_geolocation_base.withColumnRenamed("geolocation_zip_code_prefix", "customer_zip_code_prefix") \
                                          .withColumnRenamed("geolocation_lat", "customer_lat") \
                                          .withColumnRenamed("geolocation_lng", "customer_lng")

 Consolidação (Merge) das Bases

In [ ]:
from pyspark.sql import functions as F

# Conversão da base de itens para PySpark
df_items = spark.createDataFrame(df_raw['items'])

# Agrupamento do valor do frete por pedido
df_freight = df_items.groupBy("order_id").agg(F.sum("freight_value").alias("valor_frete"))

# Consolidação com joins explícitos para manter a integridade dos dados
df_olist_consolidado = df_pedidos \
    .join(df_clientes, "customer_id", "left") \
    .join(df_reviews, "order_id", "left") \
    .join(df_geolocation, "customer_zip_code_prefix", "left") \
    .join(df_freight, "order_id", "left")

In [ ]:
df_olist_consolidado.printSchema()

Limpeza Final, Validação e Exportação
Finalizamos removendo nulos e validando os dados antes de salvar.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType # Import DateType

# --- Etapa: Limpeza e Tratamento Final ---

# 1. Tratamento de nulos em 'review_score' (valor neutro -1)
df_olist_tratado = df_olist_consolidado.fillna({'review_score': -1})

# 2. Gestão de Geolocalização (Criação de flag e preenchimento de nulos)
# Como o CEP já foi tratado no início, aqui focamos apenas em garantir
# que os nulos de lat/lng não quebrem os cálculos futuros.
df_olist_tratado = df_olist_tratado.withColumn(
    "tem_geolocalizacao",
    F.when(F.col("customer_lat").isNotNull(), True).otherwise(False)
).fillna({'customer_lat': 0.0, 'customer_lng': 0.0})

# 3. Conversão de data (única vez)
# Padronizamos para facilitar operações temporais
# Explicitamente lidar com 'NaN' e strings vazias antes de converter para data

# Define a regex pattern for a valid timestamp string (e.g., 'YYYY-MM-DD HH:MM:SS')
valid_timestamp_pattern = "^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}$"

df_olist_tratado = df_olist_tratado.withColumn(
    "data",
    F.when(
        (F.col("order_delivered_customer_date").isNull()) | # Handle actual nulls
        (F.col("order_delivered_customer_date") == "NaN") | # Handle 'NaN' strings explicitly
        (F.col("order_delivered_customer_date") == "") |    # Handle empty strings
        (~F.col("order_delivered_customer_date").rlike(valid_timestamp_pattern)), # Handle strings not matching pattern
        F.lit(None).cast(DateType()) # If any of above, cast NULL to DateType
    ).otherwise(
        # For valid-looking strings, convert to timestamp first, then extract date
        F.to_date(F.to_timestamp(F.col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss"))
    )
)

# Adicionar cálculo de dt_entrega_real, dt_entrega_estimada, dias_atraso e is_gargalo
df_olist_tratado = df_olist_tratado \
    .withColumn("dt_entrega_real", F.col("data")) \
    .withColumn("dt_entrega_estimada", F.to_date(F.when(F.col("order_estimated_delivery_date") == 'NaN', F.lit(None)).otherwise(F.col("order_estimated_delivery_date")))) \
    .withColumn("dias_atraso", F.datediff(F.col("dt_entrega_real"), F.col("dt_entrega_estimada"))) \
    .withColumn("is_gargalo", F.when(F.col("dias_atraso") > 0, 1).otherwise(0))

# Renomear customer_state para estado
df_olist_tratado = df_olist_tratado.withColumnRenamed("customer_state", "estado")

# 4. Auditoria de Qualidade
print("--- Auditoria do Pipeline: Finalizada ---")
print(f"Total de registros: {df_olist_tratado.count()}")
print("Dados prontos para análise e visualização.")

In [ ]:
# Exportação para Parquet (usando PySpark)
df_olist_tratado.write.mode("overwrite").parquet(os.path.join(base_path, 'df_completo_final.parquet'))
print("✅ Pipeline concluído com sucesso!")

## 🌤️3.Ingestão e Tratamento de Dados Meteorológicos (INMET)
> O objetivo desta etapa é carregar o histórico climático, tratar valores nulos de precipitação (chuva) e preparar a base para cruzamento temporal com os momentos de compra e entrega.

In [ ]:
# Caminho completo e direto
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

In [ ]:
# 1. Carregamento dos dados brutos do INMET
# Forçamos a leitura como STRING para otimizar o processo e evitar erros de inferência de esquema.
df_southeast = spark.read.option("header", "true").csv(BASE_PATH + 'southeast.csv')

#2. Verificando a estrutura
# Exibe o esquema e uma amostra para validar o carregamento inicial.
print("--- Estrutura do DataFrame Southeast ---")
df_southeast.printSchema()
df_southeast.show(5)

Conversão de Tipos e Normalização
Esta etapa é crucial para garantir a integridade dos dados antes da limpeza. Convertendo as colunas numéricas e de data agora, evitamos erros de sintaxe e inconsistências durante os cálculos matemáticos e o cruzamento de informações (JOIN) com outras bases

In [ ]:
from pyspark.sql.functions import to_date, to_timestamp, col

# Convertendo a coluna 'Data' para formato de data real
# Certifique-se de que o formato da string ('yyyy-MM-dd') corresponde ao que está no seu CSV
df_southeast = df_southeast.withColumn("Data", to_date(col("Data"), "yyyy-MM-dd"))

In [ ]:
from pyspark.sql.functions import when, col

# --- Início da correção ---
# Renomeia colunas com pontos ('.') substituindo por underscores ('_')
# Isso evita o erro de 'UNRESOLVED_COLUMN' que o Spark lança quando encontra '.' em nomes de coluna.

# Pega a lista original de colunas
original_columns = df_southeast.columns

# Cria um mapeamento para renomear colunas
rename_mapping = {}
for c in original_columns:
    if '.' in c:
        new_c = c.replace('.', '_')
        rename_mapping[c] = new_c
    else:
        rename_mapping[c] = c

# Aplica o renomeamento ao DataFrame
for old_name, new_name in rename_mapping.items():
    if old_name != new_name:
        df_southeast = df_southeast.withColumnRenamed(old_name, new_name)

# Atualiza a lista de colunas para limpar com os novos nomes (se necessário).
# Como as colunas em 'cols_para_limpar' não contêm pontos, elas permanecerão as mesmas.
cols_para_limpar_effective = [rename_mapping[c] for c in [
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)"
]]
# --- Fim da correção ---

# Abordagem otimizada: substitui todas de uma vez no plano de execução
# Se estiver usando Spark < 3.4, use o .select() com o restante das colunas
df_southeast = df_southeast.select(
    *[when(col(c) == -9999.0, None).otherwise(col(c)).alias(c) if c in cols_para_limpar_effective else col(c)
      for c in df_southeast.columns] # Itera sobre as colunas JÁ renomeadas
)

# Salva apenas uma vez
df_southeast.write.mode("overwrite").parquet(BASE_PATH + "southeast_tratado.parquet")

📂 Tratamento e Salvamento da Base stations
A base foi carregada e salva no formato Parquet, garantindo uma estrutura otimizada para o futuro join com os dados climáticos.

In [ ]:
# 1. Ingestão Otimizada: Base Station
# Carregamos como String para evitar o custo computacional de inferência de esquema.
df_station = spark.read.option("header", "true").csv(BASE_PATH + 'stations.csv')

# Exibe o esquema inicial para confirmação dos nomes das colunas
print("--- Esquema da Base Station ---")
df_station.printSchema()

Tratamento e Normalização stations
Nesta célula, aplicamos a limpeza necessária. Utilizei a função trim() para remover espaços invisíveis que costumam causar erros em joins (cruzamentos de tabelas).

In [ ]:
# --- 2. Tratamento aprimorado da Base Station ---
from pyspark.sql.functions import col, trim

# Aplicando trim em colunas que podem conter espaços e serão usadas em joins ou análises
colunas_para_limpar = ["station_code", "state", "region", "station"]

for col_name in colunas_para_limpar:
    df_station = df_station.withColumn(col_name, trim(col(col_name)))

# Opcional: Converter colunas de coordenadas para tipo numérico, se necessário
df_station = df_station.withColumn("longitude", col("longitude").cast("double"))
df_station = df_station.withColumn("latitude", col("latitude").cast("double"))

print("✅ Base Station tratada e normalizada com sucesso!")
df_station.show(5)

In [ ]:
# 2. Salvar as estações tratadas
df_station.write.mode("overwrite").parquet(BASE_PATH + "stations_tratado.parquet")
print("✅ Estações tratadas e salvas como 'stations_tratado.parquet'!")

In [ ]:
# 1. Carrega a base do INMET que já foi tratada anteriormente
df_inmet_tratado = spark.read.parquet(BASE_PATH + "southeast_tratado.parquet")

# RENOMEAR E CONVERTER TIPO DA COLUNA DE PRECIPITAÇÃO E DE DATA
# Renomeia a coluna de precipitação para um formato mais amigável e consistente
df_inmet_tratado = df_inmet_tratado.withColumnRenamed(
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)", "precipitacao_total_horario_mm"
)
# Converte a coluna de precipitação para Double para permitir cálculos de média e máximo
df_inmet_tratado = df_inmet_tratado.withColumn(
    "precipitacao_total_horario_mm",
    F.col("precipitacao_total_horario_mm").cast("double")
)
# Renomeia a coluna 'Data' para 'data' para consistência no agrupamento
df_inmet_tratado = df_inmet_tratado.withColumnRenamed("Data", "data")

# 2. Pré-processamento df_stations:
# Garante que 'state' seja renomeada para 'station_state' e 'station' para 'station_name'.
df_stations_prepped = df_station \
    .withColumnRenamed("state", "station_state") \
    .withColumnRenamed("station", "station_name") \
    .select("station_code", "station_name", "station_state")

# 3. Junção (Join) para criar uma base única do INMET (dados horários)
# Usando o arquivo já tratado em vez do 'df_weather_data_prepped' bruto
df_inmet_hourly = df_inmet_tratado.join(df_stations_prepped, on="station_code", how="left")

# 4. Agrega os dados do INMET por 'station_state' e 'data' para obter médias diárias/máximas
df_inmet_final_aggregated = df_inmet_hourly \
    .groupBy("station_state", "data") \
    .agg(
        F.avg("precipitacao_total_horario_mm").alias("media_chuva_mm"),
        F.max("precipitacao_total_horario_mm").alias("max_chuva_mm")
    )

# Exibe o resultado para conferência
df_inmet_final_aggregated.show(5)

In [ ]:
# Verifica se as colunas da base de estações apareceram no join
df_inmet_hourly.printSchema()

# Verifica se existem valores nulos na coluna de estado após o join
# Se houver muitos nulos, é sinal de que o 'station_code' não está batendo bem
df_inmet_hourly.filter(col("station_state").isNull()).show(5)

In [ ]:
# Isso mostra se o agrupamento por estado e data está correto
df_inmet_final_aggregated.show(10)

# Verifica se o número de linhas faz sentido (espera-se que seja bem menor que a base horária)
print(f"Total de registros após agregação: {df_inmet_final_aggregated.count()}")

In [ ]:
print(f"Linhas antes do join: {df_inmet_tratado.count()}")
print(f"Linhas após o join: {df_inmet_hourly.count()}")

In [ ]:
# 4. Salvamento do arquivo UNIFICADO e AGREGADO
df_inmet_final_aggregated.write.mode("overwrite").parquet(BASE_PATH + 'inmet_preparado.parquet')

print("✅ Base INMET unificada, processada e agregada com sucesso!")


## 🗺️4.Ingestão e Tratamento de Dados de Malha Rodoviária (OpenStreetMap)
> Nesta etapa, mapeamos as coordenadas geográficas (latitude e longitude) dos clientes e vendedores para calcular as distâncias reais das rotas e entender o impacto do tráfego e das rodovias no tempo de entrega.

In [ ]:
### 1. Ingestão Otimizada: Malha Rodoviária
# Carregamos apenas as colunas essenciais para o cruzamento com bases de pedidos e clima.
# Isso economiza memória RAM drasticamente.

BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/' # Garante que BASE_PATH esteja definida

cols_necessarias = ["u", "v", "length", "highway", "speed_kph", "travel_time", "estado"] # Colunas presentes na base salva

df_malha_rodoviaria = spark.read.option("header", "true") \
    .parquet(BASE_PATH + 'malha_rodoviaria_particionada/') \
    .select(cols_necessarias)

print("✅ Base Malha Rodoviária carregada com seleção de colunas otimizada.")

In [ ]:
# --- Leitura e Tratamento da Malha Rodoviária ---
# Incluímos 'estado' na seleção para que ele esteja disponível para o join.
cols_malha = ["u", "v", "length", "highway", "speed_kph", "travel_time", "estado"]

# Carrega e seleciona apenas as colunas disponíveis
df_malha_rodoviaria = spark.read.option("header", "true").parquet(BASE_PATH + 'malha_rodoviaria_particionada') \
    .select(cols_malha)

# Caso precise tratar colunas numéricas (como velocidade)
from pyspark.sql.functions import col
df_malha_rodoviaria = df_malha_rodoviaria.withColumn("speed_kph", col("speed_kph").cast("double"))

# Salva o checkpoint com a coluna 'estado' incluída
df_malha_rodoviaria.write.mode("overwrite").parquet(BASE_PATH + 'malha_rodoviaria_tratada.parquet')

print("✅ Base Malha Rodoviária salva com sucesso e com a coluna 'estado' incluída!")

In [ ]:
# Rode isso para ter certeza absoluta:
df_malha_rodoviaria.printSchema()
df_malha_rodoviaria.select("highway").distinct().show()

In [ ]:
from pyspark.sql.functions import col

# 1. Carregamento (O que você já tem)
# Corrigindo para carregar o arquivo Parquet, que é o formato correto e existente
df_malha_rodoviaria = spark.read.option("header", "true").parquet(BASE_PATH + 'malha_rodoviaria_particionada')

# 2. Conversão de tipos
df_malha_tratada = df_malha_rodoviaria.withColumn("speed_kph", col("speed_kph").cast("double")) \
                                      .withColumn("length", col("length").cast("double"))

# 3. LIMPEZA (O que falta para garantir a qualidade)
# Remove linhas onde os pontos 'u' ou 'v' são nulos (essenciais para o join)
df_malha_tratada = df_malha_tratada.dropna(subset=["u", "v"])

# Remove duplicatas caso existam segmentos repetidos
df_malha_tratada = df_malha_tratada.dropDuplicates()

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Converta o PySpark DataFrame para Pandas DataFrame para usar com geopandas
df_malha_tratada = df_malha_tratada.toPandas()

# 1. Limpa os nomes das colunas (remove espaços extras nas pontas)
df_malha_tratada.columns = df_malha_tratada.columns.str.strip()

# Adiciona a coluna 'geometry' a partir de 'u' e 'v'.
# Assumindo que 'u' e 'v' representam longitude e latitude, respectivamente.
# Se 'u' e 'v' são IDs de nós e não coordenadas, esta abordagem criará geometrias de ponto em coordenadas inválidas (com base nos IDs).
# Para criar geometrias de linha para segmentos de estrada, seriam necessários os dados de nós com coordenadas (latitude e longitude).
# No entanto, para resolver o erro 'Unknown column geometry' e seguir o fluxo, esta é a forma mais direta.

df_malha_tratada['geometry'] = gpd.points_from_xy(df_malha_tratada['u'], df_malha_tratada['v'])

# 2. Agora garante que é um GeoDataFrame
df_malha_tratada = gpd.GeoDataFrame(df_malha_tratada, geometry='geometry')

# 3. Correção de geometria
df_malha_tratada['geometry'] = df_malha_tratada.geometry.buffer(0)

print("Sucesso! Coluna 'geometry' limpa e geometria validada.")

In [ ]:
# 1. Converter para tipos otimizados antes de salvar
# Isso reduz o tamanho do arquivo em disco e a carga de I/O
df_malha_tratada = df_malha_tratada.convert_dtypes()

# 2. Salvar com compressão 'snappy' (rápida e eficiente)
# O parâmetro 'index=False' evita salvar colunas desnecessárias
# Se a base for muito grande, usamos 'partition_cols' (ex: por estado)
print("Iniciando salvamento otimizado...")

df_malha_tratada.to_parquet(
    'malha_rodoviaria_tratada.parquet',
    index=False,
    engine='pyarrow',
    compression='snappy'
)

print("✅ Base salva com sucesso e de forma rápida!")

In [ ]:
# Lista as colunas do seu DataFrame atual
print(df_malha_tratada.columns)

## Integração de Dados Logísticos (Data Mart)

In [ ]:
# Verificação das colunas em cada DataFrame
print("--- Colunas na base de Pedidos ---")
df_pedidos.printSchema()

print("\n--- Colunas na base de INMET ---")
df_inmet_final_aggregated.printSchema()

print("\n--- Colunas na base de Malha Rodoviária ---")
df_malha_rodoviaria.printSchema()

In [ ]:
from pyspark.sql import functions as F

# 1. Carregando as bases tratadas (manter padronizado)
df_pedidos = spark.read.parquet(base_path + 'df_completo_final.parquet')
df_inmet = spark.read.parquet(base_path + 'inmet_preparado.parquet')
df_malha = spark.read.parquet(base_path + 'malha_rodoviaria_tratada.parquet')


In [ ]:
# 2. Ajuste de chaves para o join (exemplo)
df_pedidos = df_pedidos.withColumnRenamed("customer_state", "estado")


In [ ]:
# 3. Join Otimizado
# Unindo Pedidos com INMET (Clima) e depois com Malha Rodoviária

# Renomear 'station_state' para 'estado' em df_inmet para compatibilidade com a chave de join
df_inmet_renamed = df_inmet.withColumnRenamed("station_state", "estado")

df_integrado = df_pedidos.join(df_inmet_renamed, on=['data', 'estado'], how='left') \
                         .join(df_malha, on=['estado'], how='left')

In [ ]:
#4. Seleção e Renomeação para o Data Mart
df_final = df_integrado.select(
    F.col("order_id").alias("id_pedido"),
    F.col("estado"),
    F.col("order_purchase_timestamp"),
    F.col("order_delivered_customer_date"),
    F.col("dias_atraso").alias("tempo_entrega"),
    F.col("media_chuva_mm").alias("precipitacao"),
    F.col("speed_kph").alias("velocidade_kmh"),
    F.col("travel_time")
)

In [ ]:
# --- 5. Exportação via Checkpoint Intermediário ---

# 1. Forçamos a escrita em um formato temporário simples (CSV ou Parquet)
# Isso corta a árvore de processamento do Spark definitivamente
caminho_temp = base_path + 'temp_processamento_checkpoint'

# Salvamos o df_final temporariamente para "limpar" o grafo do Spark
df_final.write.mode("overwrite").parquet(caminho_temp)

# 2. Lemos o arquivo recém-salvo (que agora é uma tabela física, sem histórico de joins)
df_limpo = spark.read.parquet(caminho_temp)

# 3. Agora salvamos o seu Data Mart final
caminho_final = base_path + 'data_mart_looker_final.parquet'
df_limpo.write.mode("overwrite").parquet(caminho_final)

print("✅ Data Mart salvo com sucesso após Checkpoint!")

## 🚚 5. Engenharia de Atributos (Feature Engineering): Métricas de SLA, e Análise de Correlação.
Nesta etapa, criamos as métricas de performance para avaliar o desempenho logístico e validamos, atráves de uma análise estatística, se variáveis como pluviosidade, transito...

In [ ]:
from pyspark.sql.functions import col, datediff, to_date, when
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# --- A. Engenharia de Atributos: Métricas de SLA ---
# Criamos o indicador 'is_gargalo' e calculamos os dias de atraso para monitorar a eficiência
df_analise = df_final_organizado \
    .withColumn("dt_entrega_real", to_date(col("order_delivered_customer_date"))) \
    .withColumn("dt_entrega_estimada", to_date(col("order_estimated_delivery_date"))) \
    .withColumn("dias_atraso", datediff(col("dt_entrega_real"), col("dt_entrega_estimada"))) \
    .withColumn("is_gargalo", when(col("dias_atraso") > 0, 1).otherwise(0))

# --- B. Análise de Correlação Multivariada ---
# Selecionamos variáveis chave:
# 1. 'media_chuva_mm': Impacto climático
# 2. 'velocidade_kmh': Impacto do trânsito/logística
# 3. 'dias_atraso': Variável dependente principal
# 4. 'review_score': Satisfação do cliente frente ao serviço prestado
colunas_analise = ['media_chuva_mm', 'velocidade_kmh', 'dias_atraso', 'review_score']

# Agrupamento de colunas em um vetor para processamento no Spark ML
assembler = VectorAssembler(inputCols=colunas_analise, outputCol="features_vector", handleInvalid="skip")
df_vetor = assembler.transform(df_analise)

# Cálculo da matriz de correlação (Pearson)
matriz_spark = Correlation.corr(df_vetor, "features_vector").head()[0]

# --- C. Visualização de Diagnóstico ---
# Conversão para formato Pandas para plotagem do mapa de calor
matriz_dados = matriz_spark.toArray()
df_matriz_heatmap = pd.DataFrame(matriz_dados, columns=colunas_analise, index=colunas_analise)

plt.figure(figsize=(10, 8))
sns.heatmap(df_matriz_heatmap, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Diagnóstico de Atrasos: Correlação entre Clima, Trânsito e Performance")
plt.show()

Preparação das chaves de cruzamento
Para cruzar com o INMET e a Malha Rodoviária, precisamos garantir que o nosso df_final tenha uma coluna de referência comum (geralmente estado ou cep e data).

 Reparticionamos para garantir que o Spark distribua os dados igualmente
 Isso evita que um processador fique sobrecarregado ("Data Skew")

In [ ]:
print(f"Linhas Olist: {df_final_preparado.count()}")
print(f"Linhas Malha: {df_malha_rodoviaria.count()}")
print(f"Linhas INMET: {df_inmet_preparado.count()}")

Análise Estatística (Diagnóstico de Performance)
Primeiro, vamos entender a distribuição dos atrasos para saber se eles são pontuais ou sistêmicos.

In [ ]:
import os
# Isso vai listar todos os arquivos na pasta atual para você conferir se o seu parquet está lá
print("Arquivos na pasta atual:", os.listdir(base_path))

In [ ]:
print(type(df_inmet_preparado))

In [ ]:
# Verificação de segurança
tabelas_esperadas = ["df_final_preparado", "df_malha_rodoviaria", "df_inmet_preparado"]

for nome in tabelas_esperadas:
    try:
        # Se a variável existe, printa a contagem de linhas
        print(f"✅ {nome} existe! Total de linhas: {globals()[nome].count()}")
    except NameError:
        print(f"❌ {nome} NÃO foi encontrada na memória.")
    except Exception as e:
        print(f"⚠️ {nome} encontrada, mas deu erro: {e}")

In [ ]:
from pyspark.sql.functions import col, datediff, to_date, when, lit

# Converter o df_completo (Pandas DataFrame) para PySpark DataFrame
spark_df_completo = spark.createDataFrame(df_completo)

# Renomear colunas para consistência e criar colunas de data/SLA
spark_df_completo = spark_df_completo \
    .withColumnRenamed("customer_state", "estado") \
    .withColumnRenamed("geolocation_lat", "customer_lat") \
    .withColumnRenamed("geolocation_lng", "customer_lng")

# Recalcular métricas de SLA e colunas de data
spark_df_completo = spark_df_completo \
    .withColumn("dt_entrega_real", to_date(when(col("order_delivered_customer_date") == 'NaN', lit(None)).otherwise(col("order_delivered_customer_date")))) \
    .withColumn("dt_entrega_estimada", to_date(when(col("order_estimated_delivery_date") == 'NaN', lit(None)).otherwise(col("order_estimated_delivery_date")))) \
    .withColumn("dias_atraso", datediff(col("dt_entrega_real"), col("dt_entrega_estimada"))) \
    .withColumn("is_gargalo", when(col("dias_atraso") > 0, 1).otherwise(0)) \
    .withColumn("data", to_date(col("order_purchase_timestamp")))

# Unindo tudo o que você já tratou nas células anteriores
# Certifique-se de que esses nomes de variáveis batem com o que você já carregou
df_final_organizado = spark_df_completo \
    .join(df_malha_rodoviaria, "estado", "left") \
    .join(df_inmet_preparado, ["estado", "data"], "left")

# Agora, selecionamos o que importa
df_para_looker = df_final_organizado.select(
    "order_id", "estado", "data", "dias_atraso",
    "is_gargalo", "review_score", "customer_lat", "customer_lng",
    "speed_kph", col("media_chuva_mm").alias("precipitacao") # Usando media_chuva_mm como precipitacao
)

# Salvando
caminho = "/content/drive/MyDrive/projeto_delivery_logistica/data/final_looker.parquet"
df_para_looker.write.mode("overwrite").partitionBy("estado").parquet(caminho)

print("Tudo pronto! O arquivo foi salvo com todas as colunas juntas.")

In [ ]:
# Lista as colunas de cada DataFrame para conferirmos os nomes exatos
print("Colunas de df_final_preparado:")
print(df_completo.columns)

print("\nColunas de df_malha_rodoviaria:")
print(df_malha_rodoviaria.columns)

print("\nColunas de df_inmet_preparado:")
print(df_inmet_preparado.columns)

In [ ]:
# Preenche valores nulos em review_score, se necessário
df_final = df_final.fillna({'review_score': -1})

   ## 8. de Variáveis e Correlação
Análise estatística para validar se o aumento da pluviosidade (chuva) impacta o 'atraso_entrega_dias'.

In [ ]:
# Importar bibliotecas
import seaborn as sns
import matplotlib.pyplot as plt

# Selecionar colunas numéricas, incluindo as novas métricas de interesse
# Certifique-se de que esses nomes correspondam exatamente aos do seu df_final_completo
colunas_analise = [
    'media_chuva_mm', 'atraso_entrega_dias', 'tempo_entrega_dias',
    'review_score', 'geolocation_lat', 'geolocation_lng'
]

In [ ]:
# 1. Cria uma amostra representativa (10% dos dados)
df_amostra = df_final_completo.sample(withReplacement=False, fraction=0.1, seed=42)

# 2. Agrupa as colunas da amostra no vetor
assembler = VectorAssembler(inputCols=colunas_analise, outputCol="features_vector")
df_vetor = assembler.transform(df_amostra)

# 3. Calcula a matriz de correlação usando a amostra
matriz_spark = Correlation.corr(df_vetor, "features_vector").head()[0]

In [ ]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SparkSession # Add this import
import sys # Para sys.exit() ou similar para parar a execução

# 1. Agrupa as colunas em um único vetor (necessário para o Spark)
assembler = VectorAssembler(inputCols=colunas_analise, outputCol="features_vector", handleInvalid="skip")
df_vetor = assembler.transform(df_final_completo)

In [ ]:
# 2. Calcula a matriz de correlação usando o motor do Spark
matriz_spark = Correlation.corr(df_vetor, "features_vector").head()[0]

In [ ]:
# 3. Exibe o resultado (matriz de correlação completa)
print(matriz_spark)

In [ ]:
# 1. Converte a matriz do Spark para DataFrame do Pandas
import pandas as pd
matriz_dados = matriz_spark.toArray()
df_matriz_pandas = pd.DataFrame(matriz_dados, columns=colunas_analise, index=colunas_analise)

# 2. Criar o mapa de calor
plt.figure(figsize=(10, 8))
sns.heatmap(df_matriz_pandas, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Mapa de Calor: Correlação entre Clima, Logística e Localização")
plt.show()

#🚀 9. Análise Preditiva de Atraso Logístico (Machine Learning)
Nesta etapa final do projeto, utilizamos Machine Learning para transformar dados históricos em inteligência preditiva. Nosso objetivo é treinar um modelo capaz de antecipar a probabilidade de um pedido sofrer atraso na entrega.

Preparação: Definindo o Target e as Features
Precisamos transformar nossa variável de atraso em um rótulo binário (0 ou 1).

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import when

# 1. Criar coluna 'target': 1 se atrasou (atraso_entrega_dias > 0), 0 caso contrário
df_ml = df_final.withColumn("target", when(col("atraso_entrega_dias") > 0, 1).otherwise(0))

# 2. Selecionar as colunas (features) que alimentam o modelo
# Usaremos métricas climáticas e temporais
feature_cols = ['media_chuva_mm', 'tempo_entrega_dias', 'geolocation_lat', 'geolocation_lng']

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_model = assembler.transform(df_ml)

# Dividir em Treino (80%) e Teste (20%) para validação profissional
train_data, test_data = df_model.randomSplit([0.8, 0.2], seed=42)

In [ ]:
import pandas as pd
import os

# Defina o caminho base (o mesmo que você usou anteriormente)
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

# Carrega o arquivo final preparado para o dashboard
df_dashboard = pd.read_csv(os.path.join(BASE_PATH, 'dados_dashboard.csv'))

# Exibe as colunas para verificar se review_score, lat e lng estão lá
print("Colunas disponíveis no arquivo de dashboard:")
print(df_dashboard.columns.tolist())

# Opcional: exibe uma amostra dos dados para confirmar que não estão vazios
print("\nAmostra dos dados:")
display(df_dashboard.head())

Treinamento do Modelo (Random Forest)
O Random Forest é robusto, difícil de causar overfitting e traz ótimos resultados iniciais.

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# Configurar o modelo
rf = RandomForestClassifier(labelCol="target", featuresCol="features", numTrees=100)

# Treinar o modelo
model = rf.fit(train_data)

print("Modelo treinado com sucesso!")

Avaliação de Performance
Não basta treinar, precisamos saber se o modelo acerta. Usaremos a Matriz de Confusão (implicitamente via BinaryClassificationEvaluator).

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Fazer previsões no conjunto de teste
predictions = model.transform(test_data)

# Avaliar a precisão (Area Under ROC)
evaluator = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="rawPrediction")
auc = evaluator.evaluate(predictions)

print(f"Desempenho do Modelo (AUC): {auc:.2f}")

# Mostrar exemplos onde o modelo previu o atraso
predictions.select("target", "prediction", "probability").show(10)

In [ ]:
# Extrair as importâncias do modelo Random Forest
importances = model.featureImportances

# Mapear as importâncias com os nomes das colunas
feature_names = ['media_chuva_mm', 'tempo_entrega_dias', 'geolocation_lat', 'geolocation_lng']
feature_importance_list = list(zip(feature_names, importances))

# Exibir os resultados de forma ordenada
sorted_importances = sorted(feature_importance_list, key=lambda x: x[1], reverse=True)

print("Importância das Variáveis:")
for name, importance in sorted_importances:
    print(f"{name}: {importance:.4f}")

In [ ]:
import pandas as pd
import os

# Define o caminho base onde seus arquivos estão armazenados
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

# Carrega os DataFrames necessários como pandas DataFrames
# 'df_pedidos' provavelmente se refere ao 'df_orders' tratado
df_pedidos = pd.read_parquet(os.path.join(BASE_PATH, 'orders_cleaned.parquet'))
# 'df_clientes' provavelmente se refere ao 'df_customers' tratado
df_clientes = pd.read_parquet(os.path.join(BASE_PATH, 'customers_cleaned.parquet'))
# Carrega o DataFrame de reviews. Assumindo que o arquivo CSV original está na pasta.
try:
    df_reviews = pd.read_csv(os.path.join(BASE_PATH, 'olist_order_reviews_dataset.csv'))
except FileNotFoundError:
    print("Erro: 'olist_order_reviews_dataset.csv' não encontrado. Por favor, verifique o caminho ou nome do arquivo.")
    # Cria um DataFrame vazio para evitar que o código falhe completamente
    df_reviews = pd.DataFrame(columns=['order_id', 'review_score'])

# 1. Trazer a nota de satisfação (review_score)
# Certifique-se de que a coluna 'order_id' existe em ambos
df_final = df_pedidos.merge(df_reviews[['order_id', 'review_score']], on='order_id', how='left')

# 2. Trazer a localização (cliente_estado ou cliente_regiao)
# Se você tiver uma coluna 'customer_id' na tabela de pedidos e na tabela de clientes
df_final = df_final.merge(df_clientes[['customer_id', 'customer_state']], on='customer_id', how='left')

# 3. Preencher valores vazios, se houver
df_final['review_score'] = df_final['review_score'].fillna(0) # Nota 0 se não houver avaliação
df_final['customer_state'] = df_final['customer_state'].fillna('Não informado')

# 4. Salvar novamente para atualizar o seu arquivo no Drive
df_final.to_csv('dash - dados_dashboard.csv', index=False)


In [ ]:
# Lista de dataframes que você está manipulando
dfs = {
    "df_pedidos": df_pedidos, # Substitua pelo nome correto da sua variável
    "df_reviews": df_reviews, # Se houver uma tabela de avaliações
    "df_customers": df_customers # Se houver uma tabela de clientes com local
}

for nome, df in dfs.items():
    try:
        print(f"Colunas em {nome}: {df.columns}")
    except:
        print(f"{nome} não está carregado.")

In [ ]:
# Carregue o seu dataframe (ajuste o caminho se necessário)
df_final = spark.read.parquet("/content/drive/MyDrive/projeto_delivery_logistica/data/df_final_integrado.parquet")

# Converte para Pandas e salva como CSV
df_pandas = df_final.toPandas()
df_pandas.to_csv("/content/drive/MyDrive/projeto_delivery_logistica/data/dados_dashboard.csv", index=False)

print("Arquivo salvo como CSV no seu Drive!")

In [ ]:
import pandas as pd
import os

# Defina o caminho base (o mesmo que você usou anteriormente)
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

# Carrega o arquivo final preparado para o dashboard
df_dashboard = pd.read_csv(os.path.join(BASE_PATH, 'dados_dashboard.csv'))

# Exibe as colunas para verificar se review_score, lat e lng estão lá
print("Colunas disponíveis no arquivo de dashboard:")
print(df_dashboard.columns.tolist())

# Opcional: exibe uma amostra dos dados para confirmar que não estão vazios
print("\nAmostra dos dados:")
display(df_dashboard.head())